# Geometric depth method - interactive demo

Walks through the **camera-height triangulation** model on synthetic
and real data:

1. visualise the pixel -> distance curve,
2. test single / two-frame depth on a synthetic case where we know the answer,
3. apply it to a Mendeley clip using mask-derived bboxes.

In [ ]:
import sys, pathlib, math
ROOT = pathlib.Path('..').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, matplotlib.pyplot as plt, cv2
from sfm_yolo.src.utils.camera_calibration import CameraIntrinsics, load_camera_calibration
from sfm_yolo.src.geometry.geometric_depth import GeometricDepthEstimator
from sfm_yolo.src.utils.data_loader import MendeleyVideoDataset, mask_to_bboxes

intr = load_camera_calibration(ROOT / 'sfm_yolo' / 'configs' / 'camera_calib.yaml')
geo = GeometricDepthEstimator(intr)
print(intr)

## 1. Pixel-row -> ground distance curve

Pixels close to the horizon look very far; pixels at the bottom of the image are < 5 m away.

In [ ]:
rows = np.arange(intr.cy + 1, intr.height)
d = geo.pixel_to_distance(rows)
plt.plot(rows, d)
plt.xlabel('pixel row'); plt.ylabel('forward distance (m)')
plt.title(f'h={intr.camera_height_m} m, f={intr.fy} px')
plt.ylim(0, 80); plt.grid(True); plt.show()

## 2. Synthetic check: recover a 10 cm pothole at 10 m

We construct a bbox whose top edge falls at the row corresponding to 10 m,
and whose bottom edge corresponds to 10 m + (0.10 / tan(theta_road)).

In [ ]:
h = intr.camera_height_m
f = intr.fy
d_road = 10.0
target = 0.10
theta_road = math.atan2(h, d_road)
y_road = intr.cy + f * math.tan(theta_road)
delta_d = target / math.tan(theta_road)
y_bot = intr.cy + f * math.tan(math.atan2(h, d_road + delta_d))
bbox = (intr.cx - 100, y_road, intr.cx + 100, y_bot)
res = geo.single_frame_depth(bbox)
print(f'recovered depth = {res.depth_m*100:.2f} cm  (target 10.00 cm)')
print(f'distance to road = {res.distance_to_road_m:.2f} m, confidence = {res.confidence:.2f}')

## 3. Real Mendeley clip: bbox per frame -> depth

We sample a few frames from clip `0001`, derive bboxes from the masks,
and run the multi-frame estimator.

In [ ]:
DATA_ROOT = ROOT / 'data' / 'pothole_video' / 'pothole_video'
ds = MendeleyVideoDataset(DATA_ROOT, split='train', frame_stride=3)
frames = list(ds.iter_clip_frames(0))
boxes = []
for f in frames:
    bbs = mask_to_bboxes(f.mask, min_pixels=80)
    if bbs:
        boxes.append(max(bbs, key=lambda b: (b[2] - b[0]) * (b[3] - b[1])))
print(f'collected {len(boxes)} bboxes from {len(frames)} sampled frames')
if len(boxes) >= 2:
    res = geo.multi_frame_validation(boxes)
    print(f'multi-frame depth = {res.depth_m*100:.2f} cm  conf={res.confidence:.2f}')
    print(f'distances: road={res.distance_to_road_m:.2f} m, bottom={res.distance_to_bottom_m:.2f} m')
    print(f'per-frame depths (cm):', [f'{d*100:.1f}' for d in res.per_frame_depths])